# N11 Tape Max `dλ` Ablation

Runs the training-time max-`dλ` convergence ablation for tape using the `2d_tape_ICNN.ipynb` configuration. This identifies the largest continuation step at which each architecture can train and evaluate successfully.

In [ ]:
import json
import os
from pathlib import Path

import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

from properties import TapeN11Properties
from run_max_dlambda_ablation_minimal import (
    MaxDlambdaAblationConfig,
    run_max_dlambda_ablation,
    run_max_dlambda_hessian_diagnostics,
    summarize_best_stable_step,
)
from run_architectures import subset_tape_tube_candidates, subset_brazier_stiffness_only

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "max_dlambda_ablation_outputs_n11_tape"
train_file = "../experiment_data/tape_data/11_noded/n11_tape_train_dataset.npz"
valid_file = "../experiment_data/tape_data/11_noded/n11_tape_test_dataset.npz"
properties = TapeN11Properties(mass=-0.005)

K_init_chol = (0.2, 0.0, 0.1)
K_init_diag = (0.2, 0.1)

cfg = MaxDlambdaAblationConfig(
    der_K_diag=K_init_diag,
    der_K_chol=K_init_chol,
    hidden=(10,),
    corr_factor=0.01,
    input_mode="raw",
    only_stretching_NN=False,
    only_bending_NN=False,
    zero_reference=True,
    activation="tanh",
    mode="anisotropic",
    n_epochs=1000,
    lr=5e-2,
    weight_decay=1e-5,
    seed_list=(42,),
    valid_every=10,
    max_dlambda_values=(1e-3, 5e-3, 1e-2, 5e-2, 1e-1, 5e-1),
    iters=10,
    ls_steps=10,
    abs_tol=5e-4,
    rel_tol=1e-4,
    early_stop=True,
    train_fail_on_nonconvergence=True,
    prediction_fail_on_nonconvergence=False,
    hessian_reg_strength=1e-6,
    hessian_reg_probes=1,
    hessian_reg_seed=0,
    force_key=None,
    force_loss_strength=0.0,
    force_components=(0, 1, 2),
    force_sign=1.0,
    return_loss_components=False,
    early_stopping=True,
    early_stopping_patience=200,
    early_stopping_min_delta=1e-5,
    restore_best_model=True,
    output_dir=str(OUTPUT_DIR),
    save_npz=True,
    save_model=True,
    save_plots=True,
    save_summary_json=True,
    strict_finite_check=True,
    stop_after_first_failure=True,
    verbose=True,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving max_dlambda ablation results under: {OUTPUT_DIR.resolve()}")


In [ ]:
selected_architectures = [
    "brazier_chol_stiffness_baseline",
    "brazier_chol_stiffness_mlp",
    "brazier_chol_stiffness_icnn",
]

# For all tape/tube candidates, use:
# selected_architectures = subset_tape_tube_candidates()

print(f"Running {len(selected_architectures)} architectures:")
for name in selected_architectures:
    print(f"  - {name}")


In [ ]:
results = run_max_dlambda_ablation(
    properties=properties,
    train_file=train_file,
    valid_file=valid_file,
    cfg=cfg,
    selected_architectures=selected_architectures,
)

summary = summarize_best_stable_step(results)
print("\nLargest stable max_dlambda by architecture:")
print(json.dumps(summary, indent=2))


In [ ]:
run_max_dlambda_hessian_diagnostics(
    str(OUTPUT_DIR),
    use_predicted=True,
    splits=("train", "valid"),
    max_trajectories=1,
    stride=10,
    properties_class="TapeN11Properties",
)
print("Hessian diagnostics table:", OUTPUT_DIR / "hessian_diagnostics_table.csv")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

with open(OUTPUT_DIR / "all_results.json") as f:
    all_results = json.load(f)
rows = [r for records in all_results.values() for r in records]
df = pd.DataFrame(rows)
display(df[["arch_name", "seed", "max_dlambda", "success", "failure_reason", "final_valid_loss"]])

if not df.empty:
    fig, ax = plt.subplots(figsize=(7, 3.5), constrained_layout=True)
    for arch, g in df.groupby("arch_name"):
        ax.plot(g["max_dlambda"], g["success"].astype(float), marker="o", label=arch)
    ax.set_xscale("log")
    ax.set_xlabel("max dλ")
    ax.set_ylabel("success")
    ax.set_title("Tape max dλ convergence")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend(fontsize=8)
    fig.savefig(OUTPUT_DIR / "max_dlambda_success_summary.pdf", bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / "max_dlambda_success_summary.png", bbox_inches="tight")
    plt.show()
